In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from datasets import load_dataset

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [97]:
name = "minicpm-v_8b_discrete"
ds = load_dataset(f"Emotion-Aware-AI-Assistant/{name}", token=hf_token)
df = pd.DataFrame(ds['train'])

In [98]:
df['predicted_emotion'].value_counts()

predicted_emotion
Neutral                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 2445
Happiness                                                                                                                                                                                                                                                                                                                                                                                                                                

In [99]:
bigger = df[df['predicted_emotion'].str.len() > 9]

In [100]:
bigger['predicted_emotion'].value_counts()

predicted_emotion
I'm sorry, but I can't assist with that request.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                419
I'm sorry, but I cannot assist with that request.                                                                                                                                                                                                                                                                                                                                                                                                                 

In [101]:
refused = bigger[bigger['predicted_emotion'].str.contains('sorry', case=False) | bigger['predicted_emotion'].str.contains('cannot', case=False)]

In [102]:
len(refused)

2943

In [103]:
wrong = bigger[~bigger.index.isin(refused.index)]

In [104]:
len(wrong)

3479

In [105]:
wrong['predicted_emotion'].value_counts()

predicted_emotion
<one_emotion_label>                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      19
Not applicable                                                                                                                                                                                                                                                                                            

It refused to answer 2943 times and answered in a wrong format 3479 times.

In [106]:
df['predicted_emotion'] = ['emotion_refused' if idx in refused.index else pred for idx, pred in zip(df.index, df['predicted_emotion'])]
df['predicted_emotion'] = ['wrong_format' if idx in wrong.index else pred for idx, pred in zip(df.index, df['predicted_emotion'])]

In [107]:
emotions = df['predicted_emotion'].unique()
print(len(emotions))
for emotion in emotions:
    print(emotion)

65
wrong_format
Anger
Neutral
anger
Happiness
Surprised
emotion_refused
neutral
happiness
Sadness
Surprise
fear
Fear
Disgust
surprise
Anger.
Shock
<NONE>
Happy
Angry
<angry>
Stress
Distress
sadness
.
sad
<no_text>
>: Anger
<Neutral>
Contempt
Sad
happy
Joy
Alarm
>Neutral
joy
Surprise.
<neutral>
."
Concern
Curiosity
angry
surprised
Intensity
concern
disgust
None
Anxious
neutral>
distrust
<NO TEXT>
<NO_VOTE>
Neutral.
"
Fear>
Confusion
>
Neutral
>
Sadness
**: Anger
Anxiety
Sadness.
<NO DATA>
format.
>: Fear
Unclear


Now let's clean the ones that aren't valid emotions or are not mapped.

In [108]:
possible_emotions = df['label'].unique()
possible_emotions

array(['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness',
       'surprise'], dtype=object)

In [ ]:


TARGET_EMOTIONS = [
    'anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise'
]

EMOTION_VARIANTS = {
    'anger':      [r'anger', r'angry'],
    'disgust':    [r'disgust', r'disgusted'],
    'fear':       [r'fear', r'afraid'],
    'happiness':  [r'happy', r'happiness'],
    'neutral':    [r'neutral', r'normal'],
    'sadness':    [r'sad(ness)?'],
    'surprise':   [r'surprise(d)?']
}

COMPILED_PATTERNS = {
    emotion: re.compile(r'|'.join(variants), flags=re.IGNORECASE)
    for emotion, variants in EMOTION_VARIANTS.items()
}

def map_emotion(label: str):
    if not isinstance(label, str):
        return None

    text = label.strip().lower()

    matches = []
    for emotion, pattern in COMPILED_PATTERNS.items():
        if pattern.search(text):
            matches.append(emotion)

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        priority = ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness',
       'surprise']
        for p in priority:
            if p in matches:
                return p

    for emotion in TARGET_EMOTIONS:
        if emotion in text:
            return emotion

    return label

def normalize_emotions(df, col='predicted_emotion', new_col='normalized_emotion'):
    df[new_col] = df[col].apply(map_emotion)
    return df


df = normalize_emotions(df)

print(df)   

        emotion                                       image_path     label  \
0         anger        all_test\anger\test_test_0017_aligned.jpg     anger   
1         anger        all_test\anger\test_test_0027_aligned.jpg     anger   
2         anger        all_test\anger\test_test_0037_aligned.jpg     anger   
3         anger        all_test\anger\test_test_0042_aligned.jpg     anger   
4         anger        all_test\anger\test_test_0057_aligned.jpg     anger   
...         ...                                              ...       ...   
15334  surprise  all_test\surprise\train_train_09135_aligned.jpg  surprise   
15335  surprise  all_test\surprise\train_train_09145_aligned.jpg  surprise   
15336  surprise  all_test\surprise\train_train_09147_aligned.jpg  surprise   
15337  surprise  all_test\surprise\train_train_09151_aligned.jpg  surprise   
15338  surprise  all_test\surprise\train_train_09158_aligned.jpg  surprise   

         model_name  response_time  \
0      minicpm-v:8b      

In [110]:
emotions = df['normalized_emotion'].unique()
print(len(emotions))
for emotion in emotions:
    print(emotion)

35
wrong_format
anger
neutral
happiness
surprise
emotion_refused
sadness
fear
disgust
Shock
<NONE>
Stress
Distress
.
<no_text>
Contempt
Joy
Alarm
joy
."
Concern
Curiosity
Intensity
concern
None
Anxious
distrust
<NO TEXT>
<NO_VOTE>
"
Confusion
Anxiety
<NO DATA>
format.
Unclear


Before, we had more than 65 samples that wasn't classified. Now we have 30, which is better.

Analyzing it, we can see that all the other classes are not valid classes, so we can turn that to Nan.

In [111]:
def map_format(label: str):
    if not isinstance(label, str):
        return None

    if '<' in label or '>' in label or 'format' in label or '"' in label or "." in label or "Nan" in label:
        return 'wrong_format'
    
    return label


df['normalized_emotion'] = df['normalized_emotion'].apply(map_format)

In [112]:
emotions = df['normalized_emotion'].unique()
print(len(emotions))
for emotion in emotions:
    print(emotion)

26
wrong_format
anger
neutral
happiness
surprise
emotion_refused
sadness
fear
disgust
Shock
Stress
Distress
Contempt
Joy
Alarm
joy
Concern
Curiosity
Intensity
concern
None
Anxious
distrust
Confusion
Anxiety
Unclear


In [113]:
valid_labels = TARGET_EMOTIONS + ['wrong_format', 'emotion_refused']

def map_emotion(label: str):
    if not isinstance(label, str):
        return None
    if label in valid_labels:
        return label
    else:
        return 'wrong_emotion'


df['normalized_emotion'] = df['normalized_emotion'].apply(map_emotion)

In [114]:
emotions = df['normalized_emotion'].unique()
print(len(emotions))
for emotion in emotions:
    print(emotion)

10
wrong_format
anger
neutral
happiness
surprise
emotion_refused
sadness
fear
disgust
wrong_emotion


In [115]:
df['normalized_emotion'].value_counts()

normalized_emotion
wrong_format       3497
emotion_refused    2943
neutral            2764
happiness          2265
anger              1489
surprise           1035
sadness             897
fear                315
disgust             100
wrong_emotion        34
Name: count, dtype: int64

In [116]:
df["correct"] = (df["label"] == df["normalized_emotion"]).astype(int)

In [117]:
df.columns

Index(['emotion', 'image_path', 'label', 'model_name', 'response_time',
       'explanation', 'predicted_emotion', 'normalized_emotion', 'correct'],
      dtype='object')

In [ ]:
method_name = name.split("_")[-1]           # "discrete"
model_name = "_".join(name.split("_")[:-1])   # "minicpm-v_8b"

print(method_name)

discrete


In [126]:
df_results = df[['model_name', 'image_path', 'label', 'normalized_emotion', 'correct', 'response_time']]

In [127]:
df_results['method'] = method_name

C:\Users\FernandaBufon\AppData\Local\Temp\ipykernel_83880\2392281910.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_results['method'] = method_name


In [128]:
df_results

,model_name,image_path,label,normalized_emotion,correct,response_time,method
0,minicpm-v:8b,all_test\anger\test_test_0017_aligned.jpg,anger,wrong_format,0,4.05,discrete
1,minicpm-v:8b,all_test\anger\test_test_0027_aligned.jpg,anger,wrong_format,0,0.79,discrete
2,minicpm-v:8b,all_test\anger\test_test_0037_aligned.jpg,anger,anger,1,1.52,discrete
3,minicpm-v:8b,all_test\anger\test_test_0042_aligned.jpg,anger,wrong_format,0,1.84,discrete
4,minicpm-v:8b,all_test\anger\test_test_0057_aligned.jpg,anger,neutral,0,1.29,discrete
...,...,...,...,...,...,...,...
15334,minicpm-v:8b,all_test\surprise\train_train_09135_aligned.jpg,surprise,surprise,1,1.12,discrete
15335,minicpm-v:8b,all_test\surprise\train_train_09145_aligned.jpg,surprise,surprise,1,1.10,discrete
15336,minicpm-v:8b,all_test\surprise\train_train_09147_aligned.jpg,surprise,surprise,1,1.02,discrete
15337,minicpm-v:8b,all_test\surprise\train_train_09151_aligned.jpg,surprise,happiness,0,1.05,discrete


In [ ]:

dataset = Dataset.from_pandas(df)
dataset.push_to_hub(f"Emotion-Aware-AI-Assistant/{name}_standardized", token=hf_token)



Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/Emotion-Aware-AI-Assistant/minicpm-v_8b_discrete_standardized/commit/6cc3e8ce34d48c9d27fc037ca21113acf1b0de3a', commit_message='Upload dataset', commit_description='', oid='6cc3e8ce34d48c9d27fc037ca21113acf1b0de3a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Emotion-Aware-AI-Assistant/minicpm-v_8b_discrete_standardized', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Emotion-Aware-AI-Assistant/minicpm-v_8b_discrete_standardized'), pr_revision=None, pr_num=None)